# <span style="color:darkblue"> Project 1 </span>

# <span style="color:darkblue"> Zepeng Zhou </span>

# <span style="color:darkblue"> Introduction </span>

<font size = "4">

In this project, I scraped data from a list on IMDb that contains 215 short dramas.  <br>
The specific webpage is available at: https://www.imdb.com/list/ls548976736/.  <br>
This list includes many popular short dramas from recent years and provides essential information <br>
that helps us understand the content features, story types, production companies,  <br>
and participants involved in the short drama industry as a form of cultural production.

My data collection process consisted of two stages.

The first stage involved extracting the information displayed on the cards  <br>
of the 215 short dramas listed on the webpage, including：
- title
- year
- rating score
- review count
- starring actors

The second stage involved using Python to navigate and interact with the website.  <br>
In this step, I programmed Python to visit each short drama’s individual page and collect additional information:
- storyline
- genre
- release date
- country of origin
- platform or official site the drama was published
- production company

# <span style="color:darkblue"> Install Necessary Packages and Functions </span>

In [2]:
# This command executes the python scripts
exec(open("/Users/zhouzepeng/Desktop/课程/# 25 fall/DATASCI-530_Computing1\
 /computing_25fall/Lecture 06/scripts/import_packages.py").read())

# <span style="color:darkblue"> Running Chrome from Python </span>

In [88]:
#driver = webdriver.Chrome(ChromeDriverManager().install()),
                            
opts = Options()
driver = webdriver.Chrome(options=opts)

<font size = "5">

Define URL

In [89]:
drama_url = 'https://www.imdb.com/list/ls548976736/'

In [90]:
driver.get(drama_url)

# <span style="color:darkblue"> Step 1：Extracting Basic Information from the Website </span>

<font size = "4">

This section of the code extracts summary-level information about each short drama listed on the IMDb webpage.  <br>
Specifically, it uses Selenium to identify and parse each content card on the list page,  <br>
collecting the title, release year, rating score, number of user reviews, and starring actors.


In [ ]:
# Wait 2 seconds to ensure the webpage finishes loading
time.sleep(2)  

# Locate all short drama cards on the IMDb list page
cards = driver.find_elements(By.CLASS_NAME, "ipc-metadata-list-summary-item")

# Extract key variables from each card

rows = []

for card in cards:
    # --- Title ---
    # Extract the full title text (e.g., "1. Perfect Love") and remove the numeric prefix
    title_full = card.find_element(By.CLASS_NAME, "ipc-title__text").text
    title = title_full.split(". ", 1)[1] if ". " in title_full else title_full

    # --- Year ---
    # Extract the release year (if available) from the metadata section
    year_els = card.find_elements(By.CLASS_NAME, "dli-title-metadata-item")
    year = year_els[0].text.strip() if year_els else None

    # --- Rating ---
    # Try to extract the IMDb rating; if not available, fall back to the <strong> element
    rating_els = card.find_elements(By.CLASS_NAME, "ipc-rating-star--rating")
    if rating_els:
        rating = rating_els[0].text.strip()
    else:
        rating_try = card.find_elements(By.XPATH, './/strong')
        rating = rating_try[0].text.strip() if rating_try else None

   # --- Review count ---
    # Extract the number of user reviews (vote count); if not found, check alternative XPath
    review_els = card.find_elements(By.CLASS_NAME, "ipc-rating-star--voteCount")
    if review_els:
        review = review_els[0].text.strip()
    else:
        review_try = card.find_elements(By.XPATH, './/span[@name="nv"]')
        review = review_try[0].text.strip() if review_try else None

    # --- Stars ---
    # Collect all starring actors listed in the "title-description-credit" section
    # Join multiple actor names with commas
    star_spans = card.find_elements(By.CLASS_NAME, "title-description-credit")
    stars = ", ".join([s.find_element(By.TAG_NAME, "a").text for s in star_spans if s.find_elements(By.TAG_NAME, "a")])

    # Append the extracted data as a dictionary to the rows list
    rows.append({"title": title, "year": year, "rating": rating, "review_count": review, "stars": stars})



In [ ]:
# Convert the list of dictionaries to a DataFrame and save as CSV
df = pd.DataFrame(rows)
print(df)
df.to_csv("imdb_results1.csv", index=False, encoding="utf-8")
print("✅ 已保存为 imdb_results1.csv")

                                    title       year rating review_count  \
0                            Perfect Love       2023    7.1        (113)   
1             Billionaire CEO's Obsession       2023    7.4        (149)   
2                        Love by Contract       2024    7.5        (220)   
3    Don't Challenge the Lady Billionaire  2024–2025    6.6        (193)   
4              Will You Be My Love Again?       2024    8.2         (88)   
..                                    ...        ...    ...          ...   
210      Dear Husband, Let's Get Divorced       2024    8.3         (30)   
211      The Lady Boss Is Done Pretending       2024    7.6        (178)   
212       Return of the Abandoned Heiress       2024    6.5        (127)   
213      The Divorced Billionaire Heiress       2024    4.8       (1.1K)   
214          The Birth of the Black Lotus       2024    7.3         (21)   

                                                stars  
0       James Liddell, Clara Ca

# <span style="color:darkblue"> Step 2：Interactions with the Pages of Each Drama </span>


<font size = "4">

This section of the code is designed to collect the titles and corresponding detail-page URLs  <br>
for all short dramas listed on the IMDb collection page before deeper scraping. In other words,  <br>
this is the “navigation setup” stage of the data collection pipeline. 

The IMDb list page only shows  <br>
summary cards (each card containing the title, year, rating, etc.), but more detailed information—like  <br>
storyline, genre, release date, and production companies—exists on each drama’s individual detail page.  <br>
Therefore, before scraping those details, we first need to create a reliable mapping between each  <br>
drama’s title and its unique page link.

By extracting all pairs of (title, detail_url):

- I ensure that each title can be directly visited later,  <br>
even if the list page layout changes or pagination occurs.

- I also avoid repeated navigation or redundant lookups,  <br>
which significantly improves efficiency and minimizes the risk of being blocked by the server.



In [ ]:
## Stage 1: collect (title, detail_url) from the list page

wait = WebDriverWait(driver, 15)

title_and_urls = []
for card in cards:
    # Title (strip the numeric prefix like "1. Perfect Love" -> "Perfect Love")
    title_full = card.find_element(By.CLASS_NAME, "ipc-title__text").text
    title = title_full.split(". ", 1)[1] if ". " in title_full else title_full

    # Detail page link: the first <a> under the card that points to /title/tt...
    link_el = card.find_element(By.XPATH, './/a[contains(@href,"/title/tt")]')
    detail_url = link_el.get_attribute("href").split("?")[0]   # 去掉尾部参数
    title_and_urls.append((title, detail_url))


print(f"Prepare to enter {len(title_and_urls)} pages to etract more details...")

<font size = "4">

Then, I visited each individual short drama’s IMDb detail page to collect additional,  <br>
fine-grained information that was not available on the list view. Specifically, I extracted six major fields:

- storyline, genres, release date, country of origin, platform/official site, and production company.

The codes navigate to each drama’s page one by one, waits for it to fully render, lightly scrolls to  <br>
trigger lazy-loaded sections, and then parses the underlying HTML  <br>
using a hybrid method that combines Selenium (for dynamic elements) and BeautifulSoup (for static HTML extraction).

<font size = "5">

Challenges and How They Were Solved

<font size = "4"> 

I encountered two persistent problems during this process:

(1) Storyline Missing or Empty

Initially, storyline was often missing because IMDb’s page sometimes renders it under different HTML structures.
- In some cases, it appears under "section data-testid="Storyline"",
- In others, it exists only in compact header blocks like "plot-xl", "plot-l", or "plot".

To fix this, I built a multi-layered fallback strategy:
- First, search for a Storyline section → extract text from its inner "div class="ipc-html-content-inner-div"".
- If not found, try header areas with data-testids "plot-xl" or "plot-l".
- Finally, fall back to the simple "plot" span near the top of the page.

(2) Genre Extraction Failing

Genres were even trickier because IMDb changes their placement across templates. Early attempts  <br>
using Selenium returned only None values. The key insight was that:
- Sometimes genres appear as a list under the Storyline section (data-testid="storyline-genres").
- Other times, they appear as clickable chips near the page header (data-testid="genres").
- And occasionally, genres are only stored in a hidden JSON-LD block "script type="application/ld+json"" used for structured data.

I solved this by adding three fallbacks:
	1.	Extract from the storyline-genres list.
	2.	If missing, read from the header “genre chips.”
	3.	If still missing, parse the embedded JSON-LD for the "genre" field.

But these methods didn't work. 

I continued to discuss this with ChatGPT, and it said it might be associated with the “Asynchronous rendering issue”， <br>
which means that the webpage loads its content dynamically using JavaScript, so when a scraper (like Selenium)  <br>
tries to extract elements too early, the target data may not yet exist in the DOM.

In the end, the key improvement came from switching from Selenium’s element-based extraction to a hybrid approach  <br>
that uses BeautifulSoup to parse the full HTML source. By analyzing driver.page_source directly,  <br>
I bypassed IMDb’s asynchronous rendering issue, allowing me to consistently retrieve the storyline  <br>
and genre information that Selenium alone often missed. This not only solved the missing-value problem but also made the scraper much faster and more stable.

In [ ]:
## Stage 2: visit each detail page and extract fields


rows = []
for i, (title, url) in enumerate(title_and_urls, 1):
    driver.get(url)
    time.sleep(1.2)  # small pause to let the page settle

    # Light scroll to trigger lazy-loaded sections (helps some modules render)
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.25);")
        time.sleep(0.3)
    except:
        pass

    # Use page_source + BeautifulSoup for robust parsing
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # ---- Storyline (primary: Storyline section; fallbacks: plot-xl/plot-l/plot) ----
    storyline = None
    sec = soup.find("section", attrs={"data-testid": "Storyline"})
    if sec:
        blk = sec.find(attrs={"data-testid": "storyline-plot-summary"})
        if blk:
            inner = blk.find("div", class_=lambda c: c and "ipc-html-content-inner-div" in c)
            if inner:
                txt = inner.get_text(" ", strip=True)
                storyline = txt if txt else None
    if not storyline:
        # header area fallbacks
        for dtid in ("plot-xl", "plot-l"):
            el = soup.find(attrs={"data-testid": dtid})
            if el:
                txt = el.get_text(" ", strip=True)
                if txt:
                    storyline = txt
                    break
    if not storyline:
        el = soup.select_one('[data-testid="plot"] span')
        if el:
            txt = el.get_text(" ", strip=True)
            storyline = txt if txt else None

    # ---- Genres (BeautifulSoup; list section first, then chips; final JSON-LD fallback) ----
    genres = []

    # A) In the Storyline area: note data-testid sits on <ul>, not <li>
    g_container = soup.find(attrs={"data-testid": "storyline-genres"})
    if g_container:
        genres = [a.get_text(strip=True) for a in g_container.find_all("a") if a.get_text(strip=True)]

    # B) Fallback: chips near the title area (data-testid="genres")
    if not genres:
        chip_nodes = soup.select(
            '[data-testid="genres"] .ipc-chip__text, '
            '[data-testid="genres"] a.ipc-chip__text, '
            '[data-testid="genres"] span.ipc-chip__text'
        )
        genres = [n.get_text(strip=True) for n in chip_nodes if n.get_text(strip=True)]

    # C) Final fallback: parse JSON-LD embedded in the page (stable on IMDb)
    if not genres:
        import json
        for s in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(s.string or "")
                if isinstance(data, dict) and "genre" in data:
                    g = data["genre"]
                    genres = g if isinstance(g, list) else [g]
                    break
            except Exception:
                pass

     # Deduplicate and keep at most the first three
    seen = set()
    genres = [g for g in genres if not (g in seen or seen.add(g))][:3]

    genre1 = genres[0] if len(genres) > 0 else None
    genre2 = genres[1] if len(genres) > 1 else None
    genre3 = genres[2] if len(genres) > 2 else None

    # ---- Release Date (Selenium; details section) ----
    try:
        li = wait.until(EC.presence_of_element_located(
            (By.XPATH, '//li[@data-testid="title-details-releasedate"]')
        ))
        a_node = li.find_element(By.XPATH, './/a[contains(@class,"ipc-metadata-list-item__list-content-item")]')
        release_date = a_node.text.strip()
    except:
        release_date = None

    # ---- Country of Origin (Selenium) ----
    try:
        country_els = driver.find_elements(
            By.XPATH,
            '//li[@data-testid="title-details-origin"]//a[contains(@class,"ipc-metadata-list-item__list-content-item")]'
        )
        countries = [c.text.strip() for c in country_els if c.text and c.text.strip()]
        country_of_origin = ", ".join(countries) if countries else None
    except:
        country_of_origin = None

    # ---- Platforms / Official Sites (Selenium; keep top 3) ----
    try:
        platform_els = driver.find_elements(
            By.XPATH,
            '//li[@data-testid="details-officialsites"]//a[contains(@class,"ipc-metadata-list-item__list-content-item")]'
        )
        plats = [p.text.strip() for p in platform_els if p.text and p.text.strip()]
    except:
        plats = []
    platform1 = plats[0] if len(plats) > 0 else None
    platform2 = plats[1] if len(plats) > 1 else None
    platform3 = plats[2] if len(plats) > 2 else None

    # ---- Production Company (Selenium) ----
    try:
        comp_els = driver.find_elements(
            By.XPATH,
            '//li[@data-testid="title-details-companies"]//a[contains(@class,"ipc-metadata-list-item__list-content-item")]'
        )
        companies = [c.text.strip() for c in comp_els if c.text and c.text.strip()]
        production_company = ", ".join(companies) if companies else None
    except:
        production_company = None

    # Accumulate one record per title
    rows.append({
        "title": title,
        "detail_url": url,
        "storyline": storyline,
        "genre1": genre1, "genre2": genre2, "genre3": genre3,
        "release_date": release_date,
        "country_of_origin": country_of_origin,
        "platform1": platform1, "platform2": platform2, "platform3": platform3,
        "production_company": production_company
    })
    print(f"({i}/215) {title} ✓")

# Build the final detail DataFrame
detail_df = pd.DataFrame(rows)
print(detail_df.head())

In [ ]:
# Convert the list of dictionaries to a DataFrame and save as CSV
detail = pd.DataFrame(rows)
print(detail.head())
detail.to_csv("imdb_results2.csv", index=False, encoding="utf-8")
print("✅ 已保存为 imdb_results2.csv")

                                  title  \
0                          Perfect Love   
1           Billionaire CEO's Obsession   
2                      Love by Contract   
3  Don't Challenge the Lady Billionaire   
4            Will You Be My Love Again?   

                               detail_url  \
0  https://www.imdb.com/title/tt30429530/   
1  https://www.imdb.com/title/tt29160058/   
2  https://www.imdb.com/title/tt30036711/   
3  https://www.imdb.com/title/tt33778588/   
4  https://www.imdb.com/title/tt33454816/   

                                           storyline   genre1   genre2  \
0  Jewelry designer Ada Prescott's life hits a lo...    Drama  Romance   
1  Naomi who was born with a silver spoon in her ...   Comedy    Drama   
2  Nia's life turns upside down when Leo, a wealt...   Comedy    Drama   
3  A powerful CEO hides her identity for love, de...  Romance     None   
4  George, the heir to a wealthy family, was save...    Drama     None   

    genre3               

# <span style="color:darkblue"> Export Data and Data Quality Check </span>


In [3]:
first = pd.read_csv("imdb_results1.csv")
second = pd.read_csv("imdb_results2.csv")

<font size = "4"> 

In the process of checking the data, I found that there were some duplicate entries.  <br>
In fact, there should be only 186 dramas, but due to these duplicates, the total count reached 215.  <br>
I went back to the original webpage and discovered that the duplicates were already present there. <br>
Therefore, I removed the duplicate entries and kept only one record for each.

In [5]:
first = first.drop_duplicates(subset="title", keep="first")
second = second.drop_duplicates(subset="title", keep="first")

In [ ]:
merged_df = pd.merge(first, second, on="title", how="left")

In [7]:
merged_df.to_csv("imdb_merged_results.csv", index=False, encoding="utf-8")

<font size = "5">

Quality Check

<font size = "5">
Quality Check

<font size = "4">

Number of variables: 16

Number of observations: 186

Overall, the dataset is relatively complete, but there are a few columns with high levels of missing data <br>
 — especially for secondary or tertiary categorical variables.

 The high proportion of missing values in variables such as genre2, genre3, platform2, and platform3 <br>
 does not indicate errors or data loss but rather reflects the structural design of the dataset. <br>
 Most short dramas on IMDb are tagged with only one or two genres, and only a few titles have three. <br>
 To capture these cases, I created additional columns (genre2 and genre3) to store the second and third genre labels<br>
  when available. Similarly, while some dramas are distributed across multiple platforms, the majority appear on only one.<br>
   The variables platform2 and platform3 were included to record additional platforms when applicable.<br>
    Therefore, the large number of missing values in these columns represents expected structural sparsity <br>
    rather than missing information, and it does not affect the overall completeness or reliability of the dataset.

In [ ]:
# Create a summary DataFrame to inspect data quality
summary = pd.DataFrame({
    # Count the number of missing (NaN) values in each column
    'missing_count': merged_df.isnull().sum(),
    # Calculate the percentage of missing values in each column
    # .mean() gives the fraction of NaN values; multiply by 100 to convert to percentage
    'missing_ratio(%)': merged_df.isnull().mean().round(3)*100,
    # Count the number of unique (distinct) values in each column
    'unique_count': merged_df.nunique(),
    # Get the data type (e.g., object, int64, float64) of each column
    'dtype': merged_df.dtypes
}).reset_index().rename(columns={'index': 'variable'})

summary = summary.sort_values(by='missing_ratio(%)', ascending=False)
display(summary)

,variable,missing_count,missing_ratio(%),unique_count,dtype
14,platform3,186,100.0,0,float64
9,genre3,162,87.1,5,object
13,platform2,151,81.2,12,object
8,genre2,124,66.7,9,object
12,platform1,32,17.2,33,object
15,production_company,20,10.8,70,object
11,country_of_origin,4,2.2,9,object
2,rating,3,1.6,34,float64
3,review_count,3,1.6,128,object
10,release_date,2,1.1,162,object
